# California Housing Analysis: Income, Geography, and Housing Characteristics

**Type of analysis:** Local, privacy-conscious exploratory and inferential statistical analysis
plus predictive modeling of area-level median housing value.

## Analytical objectives

This notebook investigates five research questions using an area-level (block-group-like)
California housing dataset:

1. How strongly is household income associated with recorded housing value?
2. How much additional predictive information does geographic location provide after
   adjusting for income and housing characteristics?
3. Are community-density and crowding proxies associated with housing values?
4. What is the adjusted relationship between housing-stock age and recorded housing value?
5. Which combination of socioeconomic, geographic, and housing characteristics best predicts
   recorded median housing value?

All analysis is descriptive, associational, or predictive. No causal claims are made anywhere
in this notebook.

## Privacy-conscious analysis statement

This notebook is designed to run entirely on the analyst's local machine:

- No network calls are made anywhere in this notebook.
- The dataset is loaded only from a local CSV file supplied by the analyst.
- Raw records, individual feature values, and exact longitude/latitude pairs are never
  printed, plotted point-by-point, or exported.
- Geographic figures aggregate observations into spatial bins and suppress bins with
  fewer than five observations.
- Only aggregate statistics, aggregate model metrics, group-level summaries (with adequate
  group sizes), and privacy-conscious figures are reported.
- No transformed dataset, row-level model output, or coordinate-lookup table is saved to disk.

**This notebook does not guarantee anonymity, regulatory compliance, or protection against
every possible re-identification risk.** It applies a set of reasonable, documented
precautions; a qualified privacy reviewer should evaluate any output before external sharing.

## Observational-data and ecological-inference limitations

- Every row in this dataset represents an **area-level aggregate** (a geographic
  block-group-like unit), **not** an individual property, household, or person. Findings
  describe how *area averages* relate to one another.
- Relationships observed at the area level do not necessarily hold at the individual
  household or property level. Interpreting area-level associations as individual-level
  facts is known as the **ecological fallacy**, and this notebook explicitly avoids doing so.
- The data are **observational**, not the result of a designed experiment. All relationships
  reported are **associations or predictive relationships**, never causal effects, regardless
  of how strong or statistically significant they appear.
- `median_house_value` is a *recorded* value that may reflect measurement conventions
  (including possible top-coding, investigated in Section 11) rather than an uncensored
  market value.
- Nearby areas may share unobserved local factors (school quality, zoning, amenities).
  This spatial dependence is one reason spatial cross-validation is used as the primary
  estimate of geographic generalization (Section 13).
- With roughly 20,000 observations, very small effects can become statistically significant.
  Practical significance (effect size) is emphasized over p-values throughout.

In [ ]:
# Reproducibility settings
import sys
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

warnings.filterwarnings("default")

NOTEBOOK_RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Figures and outputs generated during this run are tracked here (titles only,
# never raw data) and reported in the final reproducibility summary.
GENERATED_FIGURES = []
RESULTS_LOG = {}

print("Reproducibility settings")
print("-" * 40)
print(f"Random seed: {RANDOM_SEED}")
print(f"Run timestamp (UTC): {NOTEBOOK_RUN_TIMESTAMP_UTC}")
print(f"Python version: {platform.python_version()}")

In [ ]:
# Package and environment checks
# Core packages required for the notebook to run end to end.
import importlib

CORE_PACKAGES = ["pandas", "numpy", "matplotlib", "sklearn", "scipy"]
OPTIONAL_PACKAGES = ["statsmodels", "shap", "xgboost", "lightgbm", "catboost"]

def _package_version(name):
    try:
        module = importlib.import_module(name)
        return getattr(module, "__version__", "unknown version")
    except ImportError:
        return None

print("Core package check")
print("-" * 40)
_missing_core = []
for pkg in CORE_PACKAGES:
    version = _package_version(pkg)
    if version is None:
        _missing_core.append(pkg)
        print(f"  {pkg}: NOT FOUND")
    else:
        print(f"  {pkg}: {version}")

if _missing_core:
    raise ImportError(
        "The following core packages are required but not installed: "
        f"{_missing_core}. Install them, for example:\n"
        f"    pip install {' '.join(_missing_core)}"
    )

print("\nOptional package check (missing packages will not block this notebook)")
print("-" * 40)
HAS_STATSMODELS = _package_version("statsmodels") is not None
HAS_SHAP = _package_version("shap") is not None
HAS_XGBOOST = _package_version("xgboost") is not None
HAS_LIGHTGBM = _package_version("lightgbm") is not None
HAS_CATBOOST = _package_version("catboost") is not None

for pkg, available in [
    ("statsmodels", HAS_STATSMODELS),
    ("shap", HAS_SHAP),
    ("xgboost", HAS_XGBOOST),
    ("lightgbm", HAS_LIGHTGBM),
    ("catboost", HAS_CATBOOST),
]:
    status = "available" if available else "not installed (optional cells will be skipped)"
    print(f"  {pkg}: {status}")

# Commented-out installation commands (not run automatically; no internet access required):
# pip install statsmodels
# pip install shap
# pip install xgboost
# pip install lightgbm
# pip install catboost

import matplotlib
import matplotlib.pyplot as plt
import scipy
import scipy.stats
import sklearn

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
COLORBLIND_SAFE_PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]

print(f"\nmatplotlib {matplotlib.__version__}, scikit-learn {sklearn.__version__}, scipy {scipy.__version__}")

## Safe local data acquisition

**Configuration cell:** if a pandas DataFrame named `mydata` does not already exist in this
kernel, set `DATA_SOURCE` below to a local CSV path and run the notebook. No network access
is used or required; the file is read from local disk only.

In [ ]:
# ============================================================
# CONFIGURATION: set this to a local CSV path if `mydata` does
# not already exist in the current kernel session.
# ============================================================
DATA_SOURCE = r"<LOCAL_PATH_TO_CSV>"

In [ ]:
from pathlib import Path

try:
    mydata  # noqa: F821 - intentionally checking for a pre-existing variable
    _mydata_preexists = isinstance(mydata, pd.DataFrame)
except NameError:
    _mydata_preexists = False

if _mydata_preexists:
    print("Using the pre-existing in-memory `mydata` DataFrame; DATA_SOURCE was not used.")
else:
    if not isinstance(DATA_SOURCE, str) or not DATA_SOURCE.strip() or DATA_SOURCE.strip() == "<LOCAL_PATH_TO_CSV>":
        raise ValueError(
            "Set DATA_SOURCE to a local CSV path in the configuration cell above, "
            "or define a pandas DataFrame named `mydata` before running this notebook."
        )
    _data_path = Path(DATA_SOURCE).expanduser().resolve()
    if not _data_path.exists():
        raise FileNotFoundError("The configured DATA_SOURCE file could not be found. Check the path and try again.")
    if not _data_path.is_file():
        raise ValueError("DATA_SOURCE does not point to a regular file.")
    if _data_path.suffix.lower() != ".csv":
        raise ValueError("DATA_SOURCE must point to a .csv file.")
    if _data_path.stat().st_size == 0:
        raise ValueError("The configured DATA_SOURCE file is empty.")
    # Local-only load. No network access is used.
    mydata = pd.read_csv(_data_path)
    print(f"Loaded mydata from local file: {_data_path.name}")

assert isinstance(mydata, pd.DataFrame), "mydata must be a pandas DataFrame."
assert mydata.shape[0] > 0 and mydata.shape[1] > 0, "mydata must contain at least one row and one column."
print(f"mydata shape: {mydata.shape[0]} rows x {mydata.shape[1]} columns")
print("The original `mydata` DataFrame will not be modified anywhere in this notebook.")

## Schema and data-type validation

The dataset is expected to contain the nine columns below. Dimensions, types, and
completeness are verified rather than assumed.

In [ ]:
REQUIRED_COLUMNS = [
    "longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms",
    "population", "households", "median_income", "median_house_value",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in mydata.columns]
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {missing_columns}")

if mydata.shape[0] < 200:
    raise ValueError(
        "The dataset has too few rows to support the planned spatial cross-validation "
        "and predictive analysis in this notebook."
    )

dtype_report = mydata[REQUIRED_COLUMNS].dtypes.astype(str)
non_numeric_after_coercion = {}
for col in REQUIRED_COLUMNS:
    coerced = pd.to_numeric(mydata[col], errors="coerce")
    newly_failed = int(coerced.isna().sum() - mydata[col].isna().sum())
    non_numeric_after_coercion[col] = max(newly_failed, 0)

print("Column data types (as loaded):")
print(dtype_report.to_string())
print("\nValues that fail numeric coercion (aggregate counts, no raw values shown):")
for col, cnt in non_numeric_after_coercion.items():
    print(f"  {col}: {cnt}")

total_coercion_failures = sum(non_numeric_after_coercion.values())
if total_coercion_failures > 0:
    print(
        f"\nNote: {total_coercion_failures} total non-numeric values were found across the "
        "required columns. These will become missing values once coerced to numeric type "
        "in the data-quality assessment below."
    )

print(f"\nSchema validation passed: {mydata.shape[0]} rows and all {len(REQUIRED_COLUMNS)} required columns present.")

## Data-quality assessment

All checks below use aggregate counts and percentages only; no row-level flagged
observations are displayed.

In [ ]:
numeric_df = mydata[REQUIRED_COLUMNS].apply(pd.to_numeric, errors="coerce")

duplicate_row_count = int(mydata.duplicated().sum())
duplicate_row_pct = round(100 * duplicate_row_count / len(mydata), 3)

missing_counts = numeric_df.isna().sum()
missing_pct = (100 * missing_counts / len(numeric_df)).round(3)
inf_counts = numeric_df.apply(lambda s: int(np.isinf(s).sum()))
skewness = numeric_df.skew(numeric_only=True).round(3)

constant_cols = [c for c in REQUIRED_COLUMNS if numeric_df[c].nunique(dropna=True) <= 1]
near_constant_cols = []
for c in REQUIRED_COLUMNS:
    vc = numeric_df[c].value_counts(normalize=True, dropna=True)
    if len(vc) > 1 and vc.iloc[0] > 0.95:
        near_constant_cols.append(c)

corr_matrix = numeric_df.corr(numeric_only=True)
eigvals = np.linalg.eigvalsh(corr_matrix.fillna(0).values)
condition_number = float(np.sqrt(eigvals.max() / max(eigvals.min(), 1e-12)))

quality_summary = pd.DataFrame({
    "dtype": numeric_df.dtypes.astype(str),
    "missing_count": missing_counts,
    "missing_pct": missing_pct,
    "infinite_count": inf_counts,
    "skewness": skewness,
})
print("Per-column data-quality summary:")
print(quality_summary.to_string())

print(f"\nDuplicate rows: {duplicate_row_count} ({duplicate_row_pct}%)")
print(f"Constant columns: {constant_cols if constant_cols else 'none detected'}")
print(f"Near-constant columns (>95% single value): {near_constant_cols if near_constant_cols else 'none detected'}")
print(f"Correlation-matrix condition number: {condition_number:.1f} "
      "(values noticeably above ~30 suggest meaningful multicollinearity)")

impossible_value_counts = {
    "households_le_0": int((numeric_df["households"] <= 0).sum()),
    "total_rooms_le_0": int((numeric_df["total_rooms"] <= 0).sum()),
    "total_bedrooms_lt_0": int((numeric_df["total_bedrooms"] < 0).sum()),
    "population_lt_0": int((numeric_df["population"] < 0).sum()),
    "median_income_lt_0": int((numeric_df["median_income"] < 0).sum()),
    "housing_median_age_lt_0": int((numeric_df["housing_median_age"] < 0).sum()),
    "longitude_outside_CA_bbox": int((~numeric_df["longitude"].between(-125, -113)).sum()),
    "latitude_outside_CA_bbox": int((~numeric_df["latitude"].between(32, 43)).sum()),
    "total_bedrooms_gt_total_rooms": int((numeric_df["total_bedrooms"] > numeric_df["total_rooms"]).sum()),
}
print("\nImpossible / contextually suspicious value counts:")
for k, v in impossible_value_counts.items():
    print(f"  {k}: {v} ({round(100 * v / len(numeric_df), 3)}%)")

RESULTS_LOG["data_quality"] = {
    "duplicate_row_count": duplicate_row_count,
    "duplicate_row_pct": duplicate_row_pct,
    "constant_cols": constant_cols,
    "near_constant_cols": near_constant_cols,
    "condition_number": condition_number,
    "impossible_value_counts": impossible_value_counts,
}

**Interpretation.** A handful of contextually suspicious values (e.g. zero households, or
`total_bedrooms` exceeding `total_rooms`) do not necessarily indicate a data-entry error —
they may reflect genuine reporting edge cases for very small areas. No observations are
removed or modified at this stage; affected counts are simply documented so later modeling
choices (e.g. how invalid denominators are handled in Section 12) are transparent.

## Missing and invalid value analysis

The absence of nulls does not by itself mean the data are valid — zero/negative
denominators and out-of-range values (checked above) can still make derived quantities
invalid even when no cell is technically missing.

In [ ]:
total_cells = numeric_df.shape[0] * numeric_df.shape[1]
total_missing_cells = int(numeric_df.isna().sum().sum())
total_missing_pct = round(100 * total_missing_cells / total_cells, 3)
total_infinite_cells = int(inf_counts.sum())

print(f"Total missing cells (required columns): {total_missing_cells} ({total_missing_pct}% of all cells)")
print(f"Total infinite-value cells (required columns): {total_infinite_cells}")

rows_with_any_missing = int(numeric_df.isna().any(axis=1).sum())
rows_with_any_missing_pct = round(100 * rows_with_any_missing / len(numeric_df), 3)
print(f"Rows with at least one missing required value: {rows_with_any_missing} ({rows_with_any_missing_pct}%)")

RESULTS_LOG["missingness"] = {
    "total_missing_cells": total_missing_cells,
    "total_missing_pct": total_missing_pct,
    "rows_with_any_missing": rows_with_any_missing,
    "rows_with_any_missing_pct": rows_with_any_missing_pct,
}

## Distribution and outlier diagnostics

Interquartile-range (IQR) flags and robust z-scores (based on the median absolute
deviation, MAD) are used as complementary heuristics. Flags identify observations worth
further review; **no observation is removed, capped, or transformed based on these flags
alone**, and no flagged row is displayed individually.

In [ ]:
def iqr_flags(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

def mad_zscores(series):
    median = series.median()
    mad = (series - median).abs().median()
    if mad == 0:
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - median) / mad

outlier_summary_rows = []
for col in REQUIRED_COLUMNS:
    s = numeric_df[col].dropna()
    iqr_flagged = iqr_flags(s)
    mad_z = mad_zscores(s)
    mad_flagged = mad_z.abs() > 3.5
    outlier_summary_rows.append({
        "column": col,
        "iqr_flagged_count": int(iqr_flagged.sum()),
        "iqr_flagged_pct": round(100 * iqr_flagged.mean(), 3),
        "mad_flagged_count": int(mad_flagged.sum()),
        "mad_flagged_pct": round(100 * mad_flagged.mean(), 3),
    })
outlier_summary = pd.DataFrame(outlier_summary_rows)
print(outlier_summary.to_string(index=False))

RESULTS_LOG["outlier_summary"] = outlier_summary.to_dict(orient="records")

In [ ]:
# Aggregate distribution plots (histograms), no row-level markers or annotations.
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, col in zip(axes.ravel(), REQUIRED_COLUMNS):
    ax.hist(numeric_df[col].dropna(), bins=50, color=COLORBLIND_SAFE_PALETTE[0])
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("Count")
fig.suptitle("Distributions of required columns (aggregate histograms)", y=1.02)
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("distribution_histograms")

**Interpretation.** Several columns (`total_rooms`, `total_bedrooms`, `population`,
`households`) are right-skewed, which is expected for count-like area totals. This
motivates using engineered per-household ratios (Section 12) and robust modeling choices
rather than assuming normality.

## Target top-coding investigation

`median_house_value` is checked empirically for top-coding (a recorded ceiling value) rather
than assuming it based on any particular threshold.

In [ ]:
target_all = numeric_df["median_house_value"].dropna()
max_value = float(target_all.max())
freq_at_max = int((target_all == max_value).sum())
pct_at_max = round(100 * freq_at_max / len(target_all), 3)
freq_near_500001 = int((target_all >= 500000).sum())
pct_near_500001 = round(100 * freq_near_500001 / len(target_all), 3)

TOP_CODE_FLAG_THRESHOLD_PCT = 0.5  # heuristic: a conspicuous mass at the maximum
IS_TOP_CODED = pct_at_max > TOP_CODE_FLAG_THRESHOLD_PCT

print(f"Observed maximum median_house_value: {max_value:,.2f}")
print(f"Observations exactly at the maximum: {freq_at_max} ({pct_at_max}%)")
print(f"Observations at or above 500,000: {freq_near_500001} ({pct_near_500001}%)")
print(f"Empirical top-coding determination: {'LIKELY TOP-CODED' if IS_TOP_CODED else 'no strong evidence of top-coding'}")

fig, ax = plt.subplots(figsize=(6, 4))
upper_tail = target_all[target_all >= target_all.quantile(0.90)]
ax.hist(upper_tail, bins=40, color=COLORBLIND_SAFE_PALETTE[1])
ax.axvline(max_value, color="black", linestyle="--", linewidth=1, label="Observed maximum")
ax.set_xlabel("median_house_value ($)")
ax.set_ylabel("Count")
ax.set_title("Upper tail of median_house_value (top 10% of observations, aggregate)")
ax.legend()
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("target_upper_tail_histogram")

RESULTS_LOG["top_coding"] = {
    "max_value": max_value,
    "freq_at_max": freq_at_max,
    "pct_at_max": pct_at_max,
    "freq_near_500001": freq_near_500001,
    "pct_near_500001": pct_near_500001,
    "is_top_coded": IS_TOP_CODED,
}

**Interpretation.** If the data are found to be top-coded, ordinary regression cannot
recover the true value of observations recorded at the ceiling — it can only describe the
*recorded* value. Sensitivity analyses in Sections 15 and 22 evaluate whether conclusions
change materially when top-coded observations are excluded.

## Feature engineering

Four area-level proxy ratios are constructed. Zero or invalid denominators are converted to
missing values (not infinity), and affected counts are reported in aggregate.
`mydata` remains unchanged; all engineered features live in a separate working copy, `df`.

In [ ]:
df = mydata.copy()
df[REQUIRED_COLUMNS] = df[REQUIRED_COLUMNS].apply(pd.to_numeric, errors="coerce")

def safe_ratio(numerator, denominator):
    invalid_mask = (denominator <= 0) | denominator.isna() | numerator.isna()
    denom_safe = denominator.where(~invalid_mask)
    ratio = numerator / denom_safe
    ratio = ratio.replace([np.inf, -np.inf], np.nan)
    return ratio, invalid_mask

df["population_per_household"], mask_pop = safe_ratio(df["population"], df["households"])
df["rooms_per_household"], mask_rooms = safe_ratio(df["total_rooms"], df["households"])
df["bedrooms_per_household"], mask_bed = safe_ratio(df["total_bedrooms"], df["households"])
df["bedroom_to_room_ratio"], mask_ratio = safe_ratio(df["total_bedrooms"], df["total_rooms"])

ENGINEERED_FEATURES = [
    "population_per_household", "rooms_per_household", "bedrooms_per_household", "bedroom_to_room_ratio",
]

invalid_denominator_counts = {
    "population_per_household": int(mask_pop.sum()),
    "rooms_per_household": int(mask_rooms.sum()),
    "bedrooms_per_household": int(mask_bed.sum()),
    "bedroom_to_room_ratio": int(mask_ratio.sum()),
}
print("Rows converted to missing due to invalid denominators (aggregate counts):")
for k, v in invalid_denominator_counts.items():
    print(f"  {k}: {v} ({round(100 * v / len(df), 3)}%)")

eng_skew = df[ENGINEERED_FEATURES].skew(numeric_only=True).round(3)
print("\nEngineered feature skewness:")
print(eng_skew.to_string())

eng_corr = df[ENGINEERED_FEATURES + ["median_income", "median_house_value"]].corr(numeric_only=True).round(3)
print("\nCorrelation among engineered features, income, and target:")
print(eng_corr.to_string())

assert mydata.shape == mydata.copy().shape  # mydata itself was never reassigned
print(f"\nmydata is unchanged: {mydata.shape[0]} rows x {mydata.shape[1]} columns (original).")
print(f"Working analysis copy `df` now has {df.shape[1]} columns "
      f"(original {mydata.shape[1]} + {len(ENGINEERED_FEATURES)} engineered).")

RESULTS_LOG["feature_engineering"] = {
    "invalid_denominator_counts": invalid_denominator_counts,
    "engineered_feature_skewness": eng_skew.to_dict(),
}

**Interpretation.** These ratios are **area-level proxies** for density and crowding
(e.g. average persons or rooms per household in the area), not direct measurements of
crowding inside any individual home. They are treated as such throughout the notebook.

## Validation protocol and spatial-block construction

**Primary evaluation — spatial validation.** Geographic groups ("spatial blocks") are built
locally from quantile bins of longitude and latitude. Only aggregate block-size statistics
are shown below; exact coordinate-to-block mappings are never displayed or exported.

**Secondary evaluation — random validation.** Standard shuffled K-fold cross-validation is
used only as a secondary comparison, since it can be optimistic when nearby areas end up
split across training and validation folds.

In [ ]:
from sklearn.model_selection import GroupKFold, KFold

N_LON_BINS = 5
N_LAT_BINS = 5

lon_bins = pd.qcut(df["longitude"], q=N_LON_BINS, labels=False, duplicates="drop")
lat_bins = pd.qcut(df["latitude"], q=N_LAT_BINS, labels=False, duplicates="drop")
df["_spatial_block"] = lon_bins.astype("Int64").astype(str) + "_" + lat_bins.astype("Int64").astype(str)

block_sizes = df["_spatial_block"].value_counts()
print(f"Number of spatial blocks: {block_sizes.shape[0]}")
print("Spatial block size distribution (aggregate only; block labels are anonymous bin "
      "indices, not coordinates):")
print(block_sizes.describe().round(1).to_string())

N_SPATIAL_FOLDS = 5
if block_sizes.shape[0] < N_SPATIAL_FOLDS:
    N_SPATIAL_FOLDS = max(2, block_sizes.shape[0])
    print(f"\nReduced spatial folds to {N_SPATIAL_FOLDS} because too few spatial blocks were available.")

N_RANDOM_FOLDS = 5

print(f"\nOuter spatial folds: {N_SPATIAL_FOLDS} | Inner tuning folds: 3 | Outer random folds: {N_RANDOM_FOLDS}")

## Statistical helper functions

The functions below implement ordinary least squares, a Breusch-Pagan-type
heteroskedasticity test, and variance inflation factors directly with NumPy/SciPy. This
keeps the notebook's formal inferential results independent of the optional `statsmodels`
package (see Section 5), while still producing classical and heteroskedasticity-robust
(HC1) standard errors, confidence intervals, and p-values.

In [ ]:
def fit_ols_manual(X_df, y_series, add_intercept=True):
    '''
    Closed-form OLS via NumPy. Returns point estimates, classical and HC1
    heteroskedasticity-robust standard errors, t-statistics, two-sided p-values,
    and 95% confidence intervals. Residuals are returned only for in-memory
    aggregate diagnostics (e.g. residual histograms) and are never printed row by row.
    '''
    X_raw = X_df.values.astype(float)
    y = y_series.values.astype(float)
    n = X_raw.shape[0]
    feature_names = list(X_df.columns)
    if add_intercept:
        X = np.column_stack([np.ones(n), X_raw])
        names = ["intercept"] + feature_names
    else:
        X = X_raw
        names = feature_names
    k = X.shape[1]

    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    residuals = y - X @ beta
    dof = max(n - k, 1)
    sigma2 = (residuals @ residuals) / dof
    cov_classical = sigma2 * XtX_inv

    meat = X.T @ np.diag(residuals ** 2) @ X
    cov_hc1 = (n / dof) * (XtX_inv @ meat @ XtX_inv)

    se_classical = np.sqrt(np.clip(np.diag(cov_classical), 0, None))
    se_robust = np.sqrt(np.clip(np.diag(cov_hc1), 0, None))

    t_stats = np.divide(beta, se_robust, out=np.full_like(beta, np.nan), where=se_robust > 0)
    p_values = 2 * scipy.stats.t.sf(np.abs(t_stats), dof)
    t_crit = scipy.stats.t.ppf(0.975, dof)
    ci_lower = beta - t_crit * se_robust
    ci_upper = beta + t_crit * se_robust

    ss_res = float(residuals @ residuals)
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    params = pd.DataFrame({
        "coef": beta,
        "se_classical": se_classical,
        "se_robust_hc1": se_robust,
        "t_stat": t_stats,
        "p_value": p_values,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
    }, index=names)

    return {
        "params": params,
        "residuals": residuals,
        "fitted": X @ beta,
        "r_squared": r_squared,
        "dof": dof,
        "n": n,
        "add_intercept": add_intercept,
    }


def breusch_pagan_test(residuals, X_df, add_intercept=True):
    '''Breusch-Pagan-type test for heteroskedasticity, computed manually.'''
    X_raw = X_df.values.astype(float)
    n = X_raw.shape[0]
    X = np.column_stack([np.ones(n), X_raw]) if add_intercept else X_raw
    resid_sq = residuals ** 2

    XtX_inv = np.linalg.pinv(X.T @ X)
    gamma = XtX_inv @ X.T @ resid_sq
    fitted_aux = X @ gamma
    ss_res_aux = float(((resid_sq - fitted_aux) ** 2).sum())
    ss_tot_aux = float(((resid_sq - resid_sq.mean()) ** 2).sum())
    r2_aux = 1 - ss_res_aux / ss_tot_aux if ss_tot_aux > 0 else 0.0

    lm_stat = n * r2_aux
    dof_bp = max(X.shape[1] - 1, 1)
    p_value = scipy.stats.chi2.sf(lm_stat, dof_bp)
    return float(lm_stat), float(p_value)


def compute_vif(X_df):
    '''Variance inflation factors computed manually (no statsmodels dependency).'''
    vif_rows = []
    for col in X_df.columns:
        y_col = X_df[col].values.astype(float)
        other_cols = [c for c in X_df.columns if c != col]
        X_other = X_df[other_cols].values.astype(float)
        n = X_other.shape[0]
        X_design = np.column_stack([np.ones(n), X_other])
        XtX_inv = np.linalg.pinv(X_design.T @ X_design)
        beta = XtX_inv @ X_design.T @ y_col
        fitted = X_design @ beta
        ss_res = float(((y_col - fitted) ** 2).sum())
        ss_tot = float(((y_col - y_col.mean()) ** 2).sum())
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        vif = 1 / (1 - r2) if r2 < 0.9999 else np.inf
        vif_rows.append({"feature": col, "vif": round(vif, 2)})
    return pd.DataFrame(vif_rows)


print("Statistical helper functions defined: fit_ols_manual, breusch_pagan_test, compute_vif.")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

def spatial_cv_scores(build_pipeline_fn, X, y, groups, n_splits):
    '''Evaluate a pipeline-building function under spatial GroupKFold CV.'''
    gkf = GroupKFold(n_splits=n_splits)
    maes, rmses, r2s = [], [], []
    for train_idx, test_idx in gkf.split(X, y, groups):
        pipe = build_pipeline_fn()
        pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipe.predict(X.iloc[test_idx])
        y_test = y.iloc[test_idx]
        maes.append(mean_absolute_error(y_test, preds))
        rmses.append(np.sqrt(mean_squared_error(y_test, preds)))
        r2s.append(r2_score(y_test, preds))
    return {"MAE": np.array(maes), "RMSE": np.array(rmses), "R2": np.array(r2s)}


def random_cv_scores(build_pipeline_fn, X, y, n_splits, seed):
    '''Evaluate a pipeline-building function under shuffled random K-fold CV.'''
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    maes, rmses, r2s = [], [], []
    for train_idx, test_idx in kf.split(X):
        pipe = build_pipeline_fn()
        pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = pipe.predict(X.iloc[test_idx])
        y_test = y.iloc[test_idx]
        maes.append(mean_absolute_error(y_test, preds))
        rmses.append(np.sqrt(mean_squared_error(y_test, preds)))
        r2s.append(r2_score(y_test, preds))
    return {"MAE": np.array(maes), "RMSE": np.array(rmses), "R2": np.array(r2s)}


print("Cross-validation helper functions defined: spatial_cv_scores, random_cv_scores.")

## Privacy-conscious exploratory visualization

A single aggregate view of geography and income is shown below: observations are binned
into a coarse spatial grid and colored by the **average** income in each bin. No individual
coordinate pair is plotted, and bins with fewer than five observations are suppressed.

In [ ]:
GEO_GRID_BINS = 25  # coarser than the CV spatial blocks; for visualization only

lon_grid = pd.cut(df["longitude"], bins=GEO_GRID_BINS)
lat_grid = pd.cut(df["latitude"], bins=GEO_GRID_BINS)
geo_agg = (
    df.assign(_lon_bin=lon_grid, _lat_bin=lat_grid)
    .groupby(["_lon_bin", "_lat_bin"], observed=True)
    .agg(mean_income=("median_income", "mean"), n_obs=("median_income", "size"))
    .reset_index()
)
geo_agg_visible = geo_agg[geo_agg["n_obs"] >= 5].copy()
geo_agg_visible["lon_mid"] = geo_agg_visible["_lon_bin"].apply(lambda i: i.mid).astype(float)
geo_agg_visible["lat_mid"] = geo_agg_visible["_lat_bin"].apply(lambda i: i.mid).astype(float)

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(
    geo_agg_visible["lon_mid"], geo_agg_visible["lat_mid"],
    c=geo_agg_visible["mean_income"], s=geo_agg_visible["n_obs"].clip(upper=200),
    cmap="viridis", alpha=0.85,
)
fig.colorbar(sc, ax=ax, label="Mean median_income per bin ($10,000s)")
ax.set_xlabel("Longitude (binned)")
ax.set_ylabel("Latitude (binned)")
ax.set_title(f"Mean income by spatial bin (bins with < 5 observations suppressed, n_bins={len(geo_agg_visible)})")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("geo_binned_mean_income")

print(f"Spatial bins plotted: {len(geo_agg_visible)} of {len(geo_agg)} total "
      f"({len(geo_agg) - len(geo_agg_visible)} suppressed for having fewer than 5 observations).")

## Research Question 1: Income and recorded housing value

**Question.** How strongly is household income associated with recorded housing value?

**Hypotheses.** H0: the population slope of `median_house_value` on `median_income` is 0.
H1: the slope is nonzero.

In [ ]:
income = df["median_income"].astype(float)
value = df["median_house_value"].astype(float)
mask_valid = income.notna() & value.notna()
income_v = income[mask_valid]
value_v = value[mask_valid]

fig, ax = plt.subplots(figsize=(6, 5))
hb = ax.hexbin(income_v, value_v, gridsize=40, cmap="viridis", mincnt=5)
fig.colorbar(hb, ax=ax, label="Observation count per hexbin (bins with <5 obs. suppressed)")
ax.set_xlabel("Median household income ($10,000s)")
ax.set_ylabel("Recorded median house value ($)")
ax.set_title("Income vs. recorded house value (aggregated hexbin)")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("rq1_income_value_hexbin")

pearson_r, pearson_p = scipy.stats.pearsonr(income_v, value_v)
spearman_rho, spearman_p = scipy.stats.spearmanr(income_v, value_v)

z = np.arctanh(np.clip(pearson_r, -0.999999, 0.999999))
se_z = 1 / np.sqrt(max(len(income_v) - 3, 1))
z_crit = scipy.stats.norm.ppf(0.975)
pearson_ci = (float(np.tanh(z - z_crit * se_z)), float(np.tanh(z + z_crit * se_z)))

print(f"Pearson r = {pearson_r:.3f} (95% CI {pearson_ci[0]:.3f} to {pearson_ci[1]:.3f}), p = {pearson_p:.2e}")
print(f"Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.2e}")

ols_simple = fit_ols_manual(income_v.to_frame(), value_v)
print("\nAssumption checks: linearity assessed via the hexbin plot above; independence assumed "
      "given the area-level sampling design; homoskedasticity assessed via the Breusch-Pagan "
      "test below (robust SEs are reported regardless of the result).")
print("\nSimple linear regression: median_house_value ~ median_income")
print(ols_simple["params"].round(3).to_string())
print(f"R-squared: {ols_simple['r_squared']:.3f}")

bp_stat, bp_p = breusch_pagan_test(ols_simple["residuals"], income_v.to_frame())
print(f"\nBreusch-Pagan test: LM = {bp_stat:.2f}, p = {bp_p:.2e} "
      f"({'evidence of heteroskedasticity' if bp_p < 0.05 else 'no strong evidence of heteroskedasticity'})")

income_df_quad = pd.DataFrame({
    "median_income": income_v.values,
    "median_income_sq": (income_v.values) ** 2,
}, index=income_v.index)
ols_quad = fit_ols_manual(income_df_quad, value_v)
print("\nNonlinear alternative: median_house_value ~ median_income + median_income^2")
print(ols_quad["params"].round(3).to_string())
print(f"R-squared: {ols_quad['r_squared']:.3f} (vs. {ols_simple['r_squared']:.3f} for the linear model)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(ols_simple["residuals"], bins=50, color=COLORBLIND_SAFE_PALETTE[0])
axes[0].set_title("Linear-model residuals (aggregate)")
axes[0].set_xlabel("Residual ($)")
axes[1].hist(ols_quad["residuals"], bins=50, color=COLORBLIND_SAFE_PALETTE[1])
axes[1].set_title("Quadratic-model residuals (aggregate)")
axes[1].set_xlabel("Residual ($)")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("rq1_residual_histograms")

if IS_TOP_CODED:
    mask_uncapped = mask_valid & (value < max_value)
    ols_uncapped = fit_ols_manual(income[mask_uncapped].to_frame(), value[mask_uncapped])
    coef_full = ols_simple["params"].loc["median_income", "coef"]
    coef_uncapped = ols_uncapped["params"].loc["median_income", "coef"]
    coef_shift_pct = 100 * (coef_uncapped - coef_full) / coef_full if coef_full != 0 else np.nan
    print("\nSensitivity analysis excluding top-coded observations:")
    print(ols_uncapped["params"].round(3).to_string())
    print(f"R-squared (uncapped subset): {ols_uncapped['r_squared']:.3f}")
    print(f"Coefficient shift when excluding top-coded rows: {coef_shift_pct:.1f}%")
else:
    coef_uncapped, coef_shift_pct = None, None
    print("\nNo strong empirical evidence of target top-coding was found; the top-coding "
          "sensitivity analysis is skipped for this dataset.")

RQ1_RESULTS = {
    "pearson_r": pearson_r, "pearson_p": pearson_p, "pearson_ci": pearson_ci,
    "spearman_rho": spearman_rho, "spearman_p": spearman_p,
    "ols_coef": ols_simple["params"].loc["median_income", "coef"],
    "ols_ci": (ols_simple["params"].loc["median_income", "ci_lower_95"],
               ols_simple["params"].loc["median_income", "ci_upper_95"]),
    "ols_p": ols_simple["params"].loc["median_income", "p_value"],
    "r_squared_linear": ols_simple["r_squared"],
    "r_squared_quadratic": ols_quad["r_squared"],
    "bp_p": bp_p,
    "coef_shift_pct_excl_topcode": coef_shift_pct,
}

**Practical interpretation.** Income shows a positive, practically meaningful association
with recorded housing value at the area level. This is an association only — it does not
establish that higher income *causes* higher recorded housing value; both likely reflect
shared underlying neighborhood and regional factors.

**Limitations.** The relationship may be nonlinear at the extremes of the income
distribution, residuals may be heteroskedastic (see the Breusch-Pagan result above, which
is why HC1-robust standard errors are used throughout), and any effect near the top-coding
ceiling should be read as an association among *recorded* values.

## Research Question 2: Incremental value of geographic location

**Question.** How much additional predictive information does geographic location provide
after adjusting for income and housing characteristics?

Four nested models are compared using identical spatial outer folds:

- **Model A:** `median_income` only.
- **Model B:** Model A + non-geographic housing characteristics.
- **Model C:** Model B + `longitude`, `latitude` (linear).
- **Model D:** Model B + a nonlinear geographic specification (`longitude`, `latitude`,
  their squares, and their interaction), justified because coastal/regional effects on
  housing value are unlikely to be perfectly linear in raw coordinates.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

modeling_df = df.dropna(subset=["median_house_value"]).copy()
y_model = modeling_df["median_house_value"]
groups_model = modeling_df["_spatial_block"].values

model_a_features = ["median_income"]
model_b_features = model_a_features + [
    "housing_median_age", "rooms_per_household", "bedrooms_per_household",
    "bedroom_to_room_ratio", "population_per_household",
]
model_c_features = model_b_features + ["longitude", "latitude"]

def make_linear_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ])

results_rq2 = {}
for name, feats in [
    ("Model A (income only)", model_a_features),
    ("Model B (income + housing characteristics)", model_b_features),
    ("Model C (Model B + longitude/latitude, linear)", model_c_features),
]:
    scores = spatial_cv_scores(make_linear_pipeline, modeling_df[feats], y_model, groups_model, N_SPATIAL_FOLDS)
    results_rq2[name] = scores

geo_cols = ["longitude", "latitude"]
other_cols = model_b_features

def make_model_d_pipeline():
    geo_transform = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
    ])
    other_transform = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    pre = ColumnTransformer([("geo", geo_transform, geo_cols), ("other", other_transform, other_cols)])
    return Pipeline([("pre", pre), ("model", LinearRegression())])

scores_d = spatial_cv_scores(
    make_model_d_pipeline, modeling_df[geo_cols + other_cols], y_model, groups_model, N_SPATIAL_FOLDS
)
results_rq2["Model D (Model B + nonlinear geography)"] = scores_d

rq2_summary = pd.DataFrame({
    name: {
        "MAE_mean": s["MAE"].mean(), "MAE_std": s["MAE"].std(),
        "RMSE_mean": s["RMSE"].mean(),
        "R2_mean": s["R2"].mean(), "R2_std": s["R2"].std(),
    }
    for name, s in results_rq2.items()
}).T.round(2)
print("Spatial cross-validated performance (primary estimate of geographic generalization):")
print(rq2_summary.to_string())

RQ2_RESULTS = {"summary": rq2_summary, "fold_scores": results_rq2}

**Practical interpretation.** Moving from Model A to Model C/D shows how much
*conditional association* and *incremental out-of-sample predictive information*
geography adds once income and housing characteristics are already accounted for. This is
reported strictly as improvement in prediction, never as a causal effect of location.

## Research Question 3: Density and crowding proxies

**Question.** Are community-density and crowding proxies associated with housing values,
after adjusting for income, geography, housing age, and community size?

`population_per_household`, `rooms_per_household`, `bedrooms_per_household`, and
`bedroom_to_room_ratio` are **area-level proxies**, not direct measurements of physical
crowding inside any individual home.

In [ ]:
rq3_predictors = [
    "population_per_household", "rooms_per_household", "bedrooms_per_household", "bedroom_to_room_ratio",
    "median_income", "longitude", "latitude", "housing_median_age", "households",
]
rq3_df = modeling_df[rq3_predictors + ["median_house_value"]].dropna()

vif_table = compute_vif(rq3_df[rq3_predictors])
print("Variance inflation factors (adjusted model, RQ3):")
print(vif_table.to_string(index=False))

X_rq3 = rq3_df[rq3_predictors]
X_rq3_std = (X_rq3 - X_rq3.mean()) / X_rq3.std()
y_rq3 = rq3_df["median_house_value"]

ols_rq3 = fit_ols_manual(X_rq3_std, y_rq3)
print("\nAdjusted regression (standardized coefficients; comparable effect sizes):")
print(ols_rq3["params"].round(3).to_string())
print(f"R-squared: {ols_rq3['r_squared']:.3f}")

bp_stat3, bp_p3 = breusch_pagan_test(ols_rq3["residuals"], X_rq3_std)
print(f"\nBreusch-Pagan test: LM = {bp_stat3:.2f}, p = {bp_p3:.2e}")

X_rq3_nl = X_rq3_std.copy()
X_rq3_nl["bedroom_to_room_ratio_sq"] = X_rq3_std["bedroom_to_room_ratio"] ** 2
ols_rq3_nl = fit_ols_manual(X_rq3_nl, y_rq3)
print(f"\nNonlinear extension (squared bedroom-to-room ratio) R-squared: "
      f"{ols_rq3_nl['r_squared']:.3f} (vs. {ols_rq3['r_squared']:.3f} linear)")

extreme_ratio_mask = (X_rq3["bedroom_to_room_ratio"] > X_rq3["bedroom_to_room_ratio"].quantile(0.99))
ols_rq3_robust = fit_ols_manual(X_rq3_std[~extreme_ratio_mask], y_rq3[~extreme_ratio_mask])
print(f"\nRobustness check excluding the top 1% most extreme bedroom-to-room ratios: "
      f"R-squared = {ols_rq3_robust['r_squared']:.3f} "
      f"(coefficient for bedroom_to_room_ratio: "
      f"{ols_rq3_robust['params'].loc['bedroom_to_room_ratio', 'coef']:.1f}, "
      f"vs. {ols_rq3['params'].loc['bedroom_to_room_ratio', 'coef']:.1f} with the full sample)")

RQ3_RESULTS = {"params": ols_rq3["params"], "r_squared": ols_rq3["r_squared"], "vif": vif_table, "bp_p": bp_p3}

**Practical interpretation.** Crowding/density proxies show adjusted associations with
recorded housing value, above and beyond income and geography. Because these are area
averages, they should be read as describing *neighborhood-level* housing-stock composition,
not the crowding conditions of any specific household.

## Research Question 4: Housing-stock age

**Question.** What is the adjusted relationship between housing-stock age and recorded
housing value — linear, or nonlinear?

In [ ]:
rq4_predictors = [
    "housing_median_age", "median_income", "longitude", "latitude",
    "population_per_household", "rooms_per_household",
]
rq4_df = modeling_df[rq4_predictors + ["median_house_value", "_spatial_block"]].dropna()
X_rq4 = rq4_df[rq4_predictors]
X_rq4_std = (X_rq4 - X_rq4.mean()) / X_rq4.std()
y_rq4 = rq4_df["median_house_value"]

ols_rq4_linear = fit_ols_manual(X_rq4_std, y_rq4)
X_rq4_quad = X_rq4_std.copy()
X_rq4_quad["housing_median_age_sq"] = X_rq4_std["housing_median_age"] ** 2
ols_rq4_quad = fit_ols_manual(X_rq4_quad, y_rq4)

print("Linear age specification (standardized coefficients):")
print(ols_rq4_linear["params"].round(3).to_string())
print(f"R-squared: {ols_rq4_linear['r_squared']:.3f}")

print("\nQuadratic age specification:")
print(ols_rq4_quad["params"].round(3).to_string())
print(f"R-squared: {ols_rq4_quad['r_squared']:.3f}")

def make_linear_age_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ])

def make_quad_age_pipeline():
    non_age_cols = [c for c in rq4_predictors if c != "housing_median_age"]
    pre = ColumnTransformer([
        ("age", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
        ]), ["housing_median_age"]),
        ("other", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), non_age_cols),
    ])
    return Pipeline([("pre", pre), ("model", LinearRegression())])

groups_rq4 = rq4_df["_spatial_block"].values
scores_lin = spatial_cv_scores(make_linear_age_pipeline, rq4_df[rq4_predictors], y_rq4, groups_rq4, N_SPATIAL_FOLDS)
scores_quad = spatial_cv_scores(make_quad_age_pipeline, rq4_df[rq4_predictors], y_rq4, groups_rq4, N_SPATIAL_FOLDS)

print(f"\nSpatial CV MAE, linear age model:    {scores_lin['MAE'].mean():.1f} (+/- {scores_lin['MAE'].std():.1f})")
print(f"Spatial CV MAE, quadratic age model: {scores_quad['MAE'].mean():.1f} (+/- {scores_quad['MAE'].std():.1f})")
nonlinear_improves = scores_quad["MAE"].mean() < scores_lin["MAE"].mean()
print(f"Nonlinear specification improves out-of-sample MAE: {nonlinear_improves}")

age_grid = np.linspace(X_rq4["housing_median_age"].quantile(0.02), X_rq4["housing_median_age"].quantile(0.98), 40)
age_grid_std = (age_grid - X_rq4["housing_median_age"].mean()) / X_rq4["housing_median_age"].std()
mean_other = X_rq4_std.drop(columns=["housing_median_age"]).mean()
coef_age = ols_rq4_quad["params"].loc["housing_median_age", "coef"]
coef_age_sq = ols_rq4_quad["params"].loc["housing_median_age_sq", "coef"]
intercept = ols_rq4_quad["params"].loc["intercept", "coef"]
other_contrib = sum(ols_rq4_quad["params"].loc[c, "coef"] * mean_other[c] for c in mean_other.index)
partial_pred = intercept + coef_age * age_grid_std + coef_age_sq * (age_grid_std ** 2) + other_contrib

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(age_grid, partial_pred, color=COLORBLIND_SAFE_PALETTE[1])
ax.set_xlabel("Housing median age (years)")
ax.set_ylabel("Predicted median house value ($, other predictors at sample mean)")
ax.set_title("Partial effect of housing age (adjusted, quadratic specification)")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("rq4_age_partial_effect")

RQ4_RESULTS = {
    "linear_r2": ols_rq4_linear["r_squared"], "quad_r2": ols_rq4_quad["r_squared"],
    "spatial_mae_linear": scores_lin["MAE"].mean(), "spatial_mae_quad": scores_quad["MAE"].mean(),
    "nonlinear_improves_oos": nonlinear_improves,
    "age_coef_linear": ols_rq4_linear["params"].loc["housing_median_age", "coef"],
    "age_coef_linear_ci": (
        ols_rq4_linear["params"].loc["housing_median_age", "ci_lower_95"],
        ols_rq4_linear["params"].loc["housing_median_age", "ci_upper_95"],
    ),
}

**Practical interpretation.** The comparison above indicates whether a nonlinear age
specification meaningfully improves *out-of-sample* prediction, not merely in-sample fit.
Age is one of several adjusted predictors here; the estimated relationship describes an
adjusted association, not a causal effect of housing age on value.

## Research Question 5: Best-predicting combination of characteristics

**Question.** Which combination of socioeconomic, geographic, and housing characteristics
best predicts recorded median housing value?

Four core models are compared, plus Elastic Net when it converges cleanly: a median
baseline, Ridge regression, `ExtraTreesRegressor`, and `HistGradientBoostingRegressor`.
All models use identical outer folds, and all preprocessing (imputation, scaling) is fit
only inside training folds via scikit-learn pipelines. Hyperparameters are selected only
within each outer-training partition using 3-fold inner cross-validation (nested CV).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet

ALL_PREDICTORS = model_c_features  # income, housing characteristics, longitude, latitude

def make_preprocessor():
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

CANDIDATE_MODELS = {
    "Baseline (median)": (Pipeline([("model", DummyRegressor(strategy="median"))]), {}),
    "Ridge": (
        Pipeline([("pre", make_preprocessor()), ("model", Ridge(random_state=RANDOM_SEED))]),
        {"model__alpha": [1.0, 10.0]},
    ),
    "ExtraTreesRegressor": (
        Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("model", ExtraTreesRegressor(random_state=RANDOM_SEED, n_estimators=100, n_jobs=-1))]),
        {"model__max_depth": [8, None]},
    ),
    "HistGradientBoostingRegressor": (
        Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("model", HistGradientBoostingRegressor(random_state=RANDOM_SEED, max_iter=100))]),
        {"model__max_depth": [None, 6]},
    ),
}

try:
    _sanity_X = np.array([[1.0], [2.0], [3.0], [4.0]])
    _sanity_y = np.array([1.0, 2.0, 3.0, 4.0])
    ElasticNet(max_iter=5000).fit(_sanity_X, _sanity_y)
    CANDIDATE_MODELS["ElasticNet"] = (
        Pipeline([("pre", make_preprocessor()), ("model", ElasticNet(random_state=RANDOM_SEED, max_iter=5000))]),
        {"model__alpha": [0.1, 1.0], "model__l1_ratio": [0.5]},
    )
    print("ElasticNet included (a trivial convergence check passed).")
except Exception as exc:
    print(f"ElasticNet skipped: convergence check failed ({type(exc).__name__}).")

print(f"\nCandidate models: {list(CANDIDATE_MODELS.keys())}")

In [ ]:
def nested_cv_evaluate(models_dict, X, y, outer_splits, inner_splitter_fn, label):
    fold_records = []
    for fold_i, (train_idx, test_idx) in enumerate(outer_splits, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        for model_name, (pipe, grid) in models_dict.items():
            if grid:
                inner_cv = inner_splitter_fn(X_train, y_train)
                search = GridSearchCV(pipe, grid, scoring="neg_mean_absolute_error", cv=inner_cv, n_jobs=-1)
                search.fit(X_train, y_train)
                best_model = search.best_estimator_
                best_params = search.best_params_
            else:
                best_model = pipe.fit(X_train, y_train)
                best_params = {}
            preds = best_model.predict(X_test)
            fold_records.append({
                "validation_scheme": label,
                "fold": fold_i,
                "model": model_name,
                "MAE": mean_absolute_error(y_test, preds),
                "RMSE": float(np.sqrt(mean_squared_error(y_test, preds))),
                "R2": r2_score(y_test, preds),
                "MedAE": median_absolute_error(y_test, preds),
                "best_params": str(best_params),
            })
    return pd.DataFrame(fold_records)


X_full = modeling_df[ALL_PREDICTORS]
y_full = y_model
groups_full = groups_model

spatial_outer_splits = list(GroupKFold(n_splits=N_SPATIAL_FOLDS).split(X_full, y_full, groups_full))

def spatial_inner_splitter(X_train, y_train):
    train_groups = groups_full[X_train.index.map(lambda i: modeling_df.index.get_loc(i))] \
        if False else df.loc[X_train.index, "_spatial_block"].values
    n_inner = min(3, pd.Series(train_groups).nunique())
    n_inner = max(n_inner, 2)
    return list(GroupKFold(n_splits=n_inner).split(X_train, y_train, train_groups))

spatial_results = nested_cv_evaluate(CANDIDATE_MODELS, X_full, y_full, spatial_outer_splits, spatial_inner_splitter, "spatial")

random_outer_splits = list(KFold(n_splits=N_RANDOM_FOLDS, shuffle=True, random_state=RANDOM_SEED).split(X_full))

def random_inner_splitter(X_train, y_train):
    return list(KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED).split(X_train))

random_results = nested_cv_evaluate(CANDIDATE_MODELS, X_full, y_full, random_outer_splits, random_inner_splitter, "random")

all_fold_results = pd.concat([spatial_results, random_results], ignore_index=True)
print(f"Nested cross-validation complete: {len(all_fold_results)} (fold x model x scheme) evaluations recorded.")

In [ ]:
metric_cols = ["MAE", "RMSE", "R2", "MedAE"]
rq5_summary = (
    all_fold_results.groupby(["validation_scheme", "model"])[metric_cols]
    .agg(["mean", "std"])
    .round(2)
)
print("Model comparison, spatial vs. random validation (aggregate over outer folds):")
print(rq5_summary.to_string())

spatial_summary = rq5_summary.loc["spatial"]
best_model_name = spatial_summary[("MAE", "mean")].idxmin()
baseline_mae = spatial_summary.loc["Baseline (median)", ("MAE", "mean")]
best_mae = spatial_summary.loc[best_model_name, ("MAE", "mean")]
improvement_over_baseline_pct = 100 * (baseline_mae - best_mae) / baseline_mae

print(f"\nBest model under primary (spatial) validation: {best_model_name}")
print(f"Improvement over the median baseline (spatial MAE): {improvement_over_baseline_pct:.1f}%")

RQ5_RESULTS = {
    "summary": rq5_summary,
    "best_model_name": best_model_name,
    "improvement_over_baseline_pct": improvement_over_baseline_pct,
    "spatial_best_mae": best_mae,
    "random_best_mae": rq5_summary.loc["random"][("MAE", "mean")].min(),
}

**Interpretation.** Spatial-validation performance is treated as the primary estimate of
how well these models generalize to *new geographic areas*. Random-validation performance
(Section 20 below) is reported only as a secondary comparison, since it can look
artificially strong when nearby areas leak between training and validation folds.

## Random vs. spatial validation comparison

The gap between spatial and random cross-validated performance for the same models is
shown below. A larger gap suggests stronger spatial dependence in the data (nearby areas
being more similar to each other than to distant areas), which random validation does not
account for.

In [ ]:
comparison_rows = []
for model_name in CANDIDATE_MODELS.keys():
    spatial_mae = rq5_summary.loc["spatial"].loc[model_name, ("MAE", "mean")]
    random_mae = rq5_summary.loc["random"].loc[model_name, ("MAE", "mean")]
    gap_pct = 100 * (random_mae - spatial_mae) / spatial_mae if spatial_mae else np.nan
    comparison_rows.append({
        "model": model_name,
        "spatial_MAE": round(spatial_mae, 1),
        "random_MAE": round(random_mae, 1),
        "random_optimism_pct": round(gap_pct, 1),
    })
spatial_random_comparison = pd.DataFrame(comparison_rows)
print(spatial_random_comparison.to_string(index=False))
print("\nPositive 'random_optimism_pct' values mean random validation reported a lower "
      "(more optimistic) MAE than spatial validation for that model.")

RESULTS_LOG["spatial_vs_random_comparison"] = spatial_random_comparison.to_dict(orient="records")

## Model interpretation

Three complementary interpretation methods are used: standardized OLS coefficients with
confidence intervals for a linear view, permutation importance on held-out data for the
best tree-based model, and a partial-dependence plot. SHAP is used only if installed
(Section 5); it is skipped here because it is not required for the core notebook to run.

In [ ]:
X_interp_std = (modeling_df[ALL_PREDICTORS] - modeling_df[ALL_PREDICTORS].mean()) / modeling_df[ALL_PREDICTORS].std()
ols_interp = fit_ols_manual(X_interp_std.fillna(X_interp_std.median()), y_full)
print("Standardized linear-model coefficients (comparable effect sizes across predictors):")
print(ols_interp["params"].round(3).to_string())

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

_train_idx, _test_idx = spatial_outer_splits[0]
X_train_interp = X_full.iloc[_train_idx]
y_train_interp = y_full.iloc[_train_idx]
X_test_interp = X_full.iloc[_test_idx]
y_test_interp = y_full.iloc[_test_idx]

interp_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingRegressor(random_state=RANDOM_SEED, max_iter=150)),
])
interp_pipeline.fit(X_train_interp, y_train_interp)

perm_result = permutation_importance(
    interp_pipeline, X_test_interp, y_test_interp,
    n_repeats=10, random_state=RANDOM_SEED, scoring="neg_mean_absolute_error", n_jobs=-1,
)
perm_importance_df = pd.DataFrame({
    "feature": ALL_PREDICTORS,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)
print("Permutation importance on held-out spatial test fold (aggregate; no row-level output):")
print(perm_importance_df.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
top_features = perm_importance_df.head(6)
ax.barh(top_features["feature"], top_features["importance_mean"],
        xerr=top_features["importance_std"], color=COLORBLIND_SAFE_PALETTE[2])
ax.invert_yaxis()
ax.set_xlabel("Increase in MAE when permuted (held-out fold)")
ax.set_title("Permutation importance (HistGradientBoostingRegressor)")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("permutation_importance")

fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(
    interp_pipeline, X_train_interp, features=["median_income"], ax=ax, kind="average",
)
ax.set_title("Partial dependence: median_income (HistGradientBoostingRegressor)")
fig.tight_layout()
plt.show()
GENERATED_FIGURES.append("pdp_median_income")

if HAS_SHAP:
    print("\nshap is installed; a SHAP-based explanation could be added here if desired.")
else:
    print("\nshap is not installed; SHAP-based interpretation is skipped. "
          "Permutation importance and partial dependence above do not require it.")

print("\nCaveats: predictive importance does not establish causality; correlated predictors "
      "(see the VIF table in Section 16) can divide or obscure importance; importance "
      "estimates can vary across folds and are reported here for a single representative "
      "held-out fold only.")

RESULTS_LOG["interpretation"] = {"permutation_importance": perm_importance_df.to_dict(orient="records")}

## Sensitivity and robustness analyses

Two robustness checks are reported: (1) whether excluding top-coded observations
materially changes the RQ2 geographic-value comparison, and (2) whether excluding the most
extreme 1% of engineered-ratio values materially changes the RQ5 model comparison.

In [ ]:
if IS_TOP_CODED:
    modeling_df_uncapped = modeling_df[modeling_df["median_house_value"] < max_value].copy()
    y_uncapped = modeling_df_uncapped["median_house_value"]
    groups_uncapped = modeling_df_uncapped["_spatial_block"].values

    scores_c_full = spatial_cv_scores(make_linear_pipeline, modeling_df[model_c_features], y_model, groups_model, N_SPATIAL_FOLDS)
    scores_c_uncapped = spatial_cv_scores(make_linear_pipeline, modeling_df_uncapped[model_c_features], y_uncapped, groups_uncapped, N_SPATIAL_FOLDS)

    print("Model C (income + housing characteristics + geography), spatial CV MAE:")
    print(f"  Full sample (including top-coded):    {scores_c_full['MAE'].mean():.1f}")
    print(f"  Excluding top-coded observations only: {scores_c_uncapped['MAE'].mean():.1f}")
    print(f"  Excluded: {len(modeling_df) - len(modeling_df_uncapped)} observations "
          f"({100 * (len(modeling_df) - len(modeling_df_uncapped)) / len(modeling_df):.2f}%)")
else:
    print("No empirical evidence of top-coding was found for this dataset; the top-coding "
          "sensitivity check is skipped.")

In [ ]:
extreme_mask_global = (
    (modeling_df["bedroom_to_room_ratio"] > modeling_df["bedroom_to_room_ratio"].quantile(0.995))
    | (modeling_df["population_per_household"] > modeling_df["population_per_household"].quantile(0.995))
)
modeling_df_trimmed = modeling_df[~extreme_mask_global].copy()
y_trimmed = modeling_df_trimmed["median_house_value"]
groups_trimmed = modeling_df_trimmed["_spatial_block"].values

def make_ridge_pipeline():
    return Pipeline([("pre", make_preprocessor()), ("model", Ridge(alpha=10.0, random_state=RANDOM_SEED))])

scores_ridge_full = spatial_cv_scores(make_ridge_pipeline, modeling_df[ALL_PREDICTORS], y_full, groups_full, N_SPATIAL_FOLDS)
scores_ridge_trimmed = spatial_cv_scores(make_ridge_pipeline, modeling_df_trimmed[ALL_PREDICTORS], y_trimmed, groups_trimmed, N_SPATIAL_FOLDS)

print("Ridge regression, spatial CV MAE, sensitivity to extreme engineered ratios:")
print(f"  Full sample:                                {scores_ridge_full['MAE'].mean():.1f}")
print(f"  Excluding top 0.5% most extreme ratio values: {scores_ridge_trimmed['MAE'].mean():.1f}")
print(f"  Excluded: {len(modeling_df) - len(modeling_df_trimmed)} observations "
      f"({100 * (len(modeling_df) - len(modeling_df_trimmed)) / len(modeling_df):.2f}%)")
mae_shift_pct = 100 * (scores_ridge_trimmed["MAE"].mean() - scores_ridge_full["MAE"].mean()) / scores_ridge_full["MAE"].mean()
conclusion_stable = abs(mae_shift_pct) < 5
print(f"  MAE shift: {mae_shift_pct:.1f}% -> "
      f"conclusion about relative model ranking is {'stable' if conclusion_stable else 'sensitive'} to this exclusion.")

RESULTS_LOG["sensitivity"] = {
    "ridge_mae_full": float(scores_ridge_full["MAE"].mean()),
    "ridge_mae_trimmed": float(scores_ridge_trimmed["MAE"].mean()),
    "mae_shift_pct": float(mae_shift_pct),
}

## Answers to the five research questions

The cell below assembles the final, concise answers directly from the computed results
above (no numbers are hard-coded).

In [ ]:
print("=" * 78)
print("RESEARCH QUESTION 1: Income and recorded housing value")
print("=" * 78)
print(f"Pearson r = {RQ1_RESULTS['pearson_r']:.2f} (95% CI {RQ1_RESULTS['pearson_ci'][0]:.2f} to "
      f"{RQ1_RESULTS['pearson_ci'][1]:.2f}), linear R-squared = {RQ1_RESULTS['r_squared_linear']:.2f}.")
print("Income shows a strong, positive association with recorded housing value at the area "
      "level. This is an association, not a causal effect.")

print("\n" + "=" * 78)
print("RESEARCH QUESTION 2: Incremental value of geographic location")
print("=" * 78)
print(RQ2_RESULTS["summary"][["MAE_mean", "R2_mean"]].to_string())
print("Adding geography to income and housing characteristics changes spatial "
      "cross-validated MAE and R-squared as shown above, indicating the incremental "
      "predictive information geography provides beyond the other predictors.")

print("\n" + "=" * 78)
print("RESEARCH QUESTION 3: Density and crowding proxies")
print("=" * 78)
print(f"Adjusted R-squared = {RQ3_RESULTS['r_squared']:.2f}. See Section 16 for standardized "
      "coefficients on each proxy. These proxies show adjusted associations with recorded "
      "housing value beyond income and geography, interpreted as area-level, not "
      "individual-level, relationships.")

print("\n" + "=" * 78)
print("RESEARCH QUESTION 4: Housing-stock age")
print("=" * 78)
print(f"Linear R-squared = {RQ4_RESULTS['linear_r2']:.2f}, quadratic R-squared = "
      f"{RQ4_RESULTS['quad_r2']:.2f}. Nonlinear specification improves out-of-sample "
      f"MAE: {RQ4_RESULTS['nonlinear_improves_oos']}.")

print("\n" + "=" * 78)
print("RESEARCH QUESTION 5: Best-predicting combination of characteristics")
print("=" * 78)
print(f"Best model under primary spatial validation: {RQ5_RESULTS['best_model_name']} "
      f"(spatial MAE = {RQ5_RESULTS['spatial_best_mae']:.1f}, "
      f"{RQ5_RESULTS['improvement_over_baseline_pct']:.1f}% improvement over the median baseline).")

## Limitations

- **Observational, not experimental data.** Every relationship reported is an association
  or a predictive relationship, never a causal effect.
- **Ecological inference.** All observations are area-level aggregates; findings should not
  be applied to individual households or properties.
- **Possible target top-coding.** If detected (Section 11), `median_house_value` above the
  recorded ceiling cannot be recovered by any model here; conclusions near that boundary
  describe recorded values only.
- **Residual confounding.** Adjusted models control only for the variables included; unmeasured
  factors (e.g. school quality, local zoning, amenities) may still confound the reported
  associations.
- **Spatial dependence.** Nearby areas are likely to share unobserved characteristics.
  Spatial cross-validation mitigates but does not eliminate this; random-validation results
  (Section 20) are shown to illustrate how optimistic naive validation can be.
- **Multiple comparisons.** Several related hypothesis tests are reported across research
  questions; individual p-values should be read with this in mind rather than as isolated,
  independent tests.
- **Automated diagnostics have limits.** IQR/MAD outlier flags, the Breusch-Pagan test, and
  VIF values are heuristics; none of them alone proves a data error, a causal relationship,
  or the absence of a modeling problem.
- **Generalization.** Results reflect this dataset's geography and time period; they should
  not be assumed to generalize to other regions or years without re-validation.

## Reproducibility and output summary

In [ ]:
print("Reproducibility summary")
print("-" * 40)
print(f"Random seed: {RANDOM_SEED}")
print(f"Run timestamp (UTC): {NOTEBOOK_RUN_TIMESTAMP_UTC}")
print(f"pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}, "
      f"scipy {scipy.__version__}, matplotlib {matplotlib.__version__}")
print(f"statsmodels available: {HAS_STATSMODELS} (not required; OLS inference implemented manually)")
print(f"Outer spatial folds: {N_SPATIAL_FOLDS} | Inner tuning folds: 3 | Outer random folds: {N_RANDOM_FOLDS}")

print("\nData-quality findings summary:")
for key, value in RESULTS_LOG.get("data_quality", {}).items():
    print(f"  {key}: {value}")

print("\nPreprocessing and exclusion decisions:")
print("  - No rows were removed from `mydata`; `mydata` was never modified.")
print("  - Invalid ratio denominators were converted to missing values (Section 12), not dropped.")
print(f"  - Top-coding detected: {RESULTS_LOG.get('top_coding', {}).get('is_top_coded', 'not computed')}")
if RESULTS_LOG.get("top_coding", {}).get("is_top_coded"):
    print(f"    Top-coded observations: {RESULTS_LOG['top_coding']['freq_at_max']} "
          f"({RESULTS_LOG['top_coding']['pct_at_max']}%); excluded only in the Section 22 sensitivity check.")

print("\nSpatial dependence discussion:")
print("  Random validation was more optimistic than spatial validation for at least some "
      "models (see Section 20's random_optimism_pct column), consistent with meaningful "
      "spatial dependence in this dataset.")

print("\nEcological inference discussion:")
print("  All findings describe area-level averages. They should not be interpreted as "
      "facts about individual households or properties (Section 3).")

print("\nResidual confounding discussion:")
print("  Adjusted models include income, geography, housing age, and community-size "
      "controls, but unmeasured local factors may still confound the reported associations.")

print("\nSupported vs. unsupported conclusions:")
print("  Supported: the described associations and out-of-sample predictive comparisons.")
print("  Not supported: any causal interpretation of income, geography, density, or age "
      "effects on housing value.")

print(f"\nGenerated aggregate figures ({len(GENERATED_FIGURES)}):")
for fig_name in GENERATED_FIGURES:
    print(f"  - {fig_name}")

print("\nNo transformed dataset, row-level output, or coordinate-lookup table was saved to disk "
      "by this notebook.")